In [0]:
# %restart_python

In [0]:
# installs for image processing
%pip install pillow
dbutils.library.restartPython()

In [0]:
# verifying the table looks correct
training_df = spark.read.table("workspace.silver.training_images")
display(training_df.select("path", "length", "label").limit(5))

In [0]:
# rezise images to 224x224 (appropiate for the model) and write to the silver table

import io
from pyspark.sql.functions import pandas_udf, col

IMAGE_RESIZE = 224

@pandas_udf("binary")
def resize_image_udf(content_series):
    def resize_image(content):
        from PIL import Image
        image = Image.open(io.BytesIO(content))
        width, height = image.size
        new_size = min(width, height)
        image = image.crop((
            (width - new_size) / 2,
            (height - new_size) / 2,
            (width + new_size) / 2,
            (height + new_size) / 2
        ))
        image = image.resize((IMAGE_RESIZE, IMAGE_RESIZE), Image.NEAREST)
        output = io.BytesIO()
        image.save(output, format="JPEG")
        return output.getvalue()
    return content_series.apply(resize_image)

training_df = spark.read.table("workspace.silver.training_images")

(training_df
    .withColumn("content", resize_image_udf(col("content")))
    .write
    .mode("overwrite")
    .saveAsTable("workspace.silver.training_images_resized")
)

In [0]:
# export to Volume as Parquet, for colab

export_path = "/Volumes/main/default/training_imgs_vol/exports/training_images_resized"

(spark.read.table("workspace.silver.training_images_resized")
    .write
    .mode("overwrite")
    .parquet(export_path)
)

print(f"Export complete: {export_path}")

In [0]:
# verify the export

display(spark.read.parquet(export_path).select("path", "label").limit(5))